# TV taste research
Reproducible exploratory analysis using a cached TVmaze snapshot. See research/output/research-report.md for conclusions and limitations. This notebook is provided unexecuted; the equivalent scripts have been run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline/jobs/analyze.py").exists()), None)
assert ROOT is not None, "Open this notebook within the project"

## Rebuild from the cached snapshot
The download step is optional while data/raw/ and data/manifest.json exist. The cached snapshot makes reruns reproducible.

In [ ]:
import subprocess, sys
# subprocess.run([sys.executable, str(ROOT / "tools/manage.py"), "job", "download"], check=True)
subprocess.run([sys.executable, str(ROOT / "tools/manage.py"), "job", "analyze"], check=True, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, str(ROOT / "tools/manage.py"), "job", "build_deliverables"], check=True)

In [ ]:
audit = json.loads((ROOT / "research/output/audit.json").read_text())
audit

## Feature prevalence
Theme flags are summary keyword proxies. Compare against other shows, not an invented negative preference class.

In [ ]:
pd.read_csv(ROOT / "research/output/feature_prevalence.csv").head(15)

In [ ]:
display(Image(filename=str(ROOT / "research/output/taste_overview.png")))
display(Image(filename=str(ROOT / "research/output/seed_theme_evidence.png")))

## Correlation
Pearson on binary indicators is phi correlation. Numeric correlations use Spearman and pairwise-complete records.

In [ ]:
pd.read_csv(ROOT / "research/output/numeric_correlations.csv", index_col=0)

In [ ]:
pd.read_csv(ROOT / "research/output/numeric_pair_counts.csv", index_col=0)

In [ ]:
display(Image(filename=str(ROOT / "research/output/theme_correlations.png")))

## Recommendations and sensitivity
Similarity scores are not probabilities. Inspect rank changes before interpreting exact order.

In [ ]:
pd.read_csv(ROOT / "research/output/recommendations.csv")[["name", "year", "score", "nearest", "loo_min", "loo_max"]].head(20)

In [ ]:
sensitivity = json.loads((ROOT / "research/output/sensitivity.json").read_text())
display(pd.DataFrame(sensitivity["variants"]))
display(pd.DataFrame(sensitivity["retrieval"]))

## Inspect the actual evidence
Theme definitions and per-seed matching terms are saved for audit. Modify pipeline/jobs/analyze.py to test different rules or favorite weights.

In [ ]:
json.loads((ROOT / "research/output/audit.json").read_text())["seed_theme_hits"]